# 계획 단계 모델 비교 — 베이스 0.6B vs 1.7B (품질)

시나리오 A의 계획 LLM(`plan_brief`)이 하는 일을 서버와 같은 프롬프트·같은 검증 규칙으로 두 모델에 시켜 본다.
**속도는 여기서 보지 않는다** (Colab GPU는 서버 CPU와 다르다). 속도는 PC에서 `run_speed`로 잰다.

준비물 (왼쪽 폴더 아이콘으로 올려도 된다):
- `app.zip` — 팀 저장소 `app/` 폴더를 묶은 것 (11/16에 올려 주신 것과 같은 판)
- `plan_eval.zip` — 이 측정 묶음 (`scripts/plan_eval/`, `tests/`)

런타임 > 런타임 유형 변경에서 GPU(T4면 충분)를 고른다.

In [ ]:
# 1) 파일 올리기: app.zip, plan_eval.zip 을 선택
import os
from google.colab import files

if not (os.path.exists('/content/app.zip') and os.path.exists('/content/plan_eval.zip')):
    files.upload()
print(sorted(f for f in os.listdir('/content') if f.endswith('.zip')))

In [ ]:
# 2) 풀기 + 의존 패키지 (transformers·torch는 Colab에 들어 있다. app 코드를 불러오는 데 pydantic 만 더 필요하다)
!rm -rf /content/repo && mkdir -p /content/repo
!cd /content/repo && unzip -q -o /content/app.zip && unzip -q -o /content/plan_eval.zip
!pip -q install pydantic pydantic-settings pytest
%cd /content/repo
!ls scripts/plan_eval

In [ ]:
# 3) 측정 코드 자체 시험 (모델 없이 돈다. 전부 통과해야 한다)
!python -B -m pytest -p no:cacheprovider tests/test_plan_eval_harness.py -q

In [ ]:
# 4) dtype 고르기: 서버는 체크포인트 dtype(auto, Qwen3는 보통 bfloat16)으로 올린다.
#    T4 같은 구형 GPU는 bfloat16을 못 쓰므로 float16으로 대신한다 (수치가 아주 조금 다를 수 있다).
import torch
assert torch.cuda.is_available(), "GPU 런타임이 아닙니다. 런타임 유형을 GPU로 바꾸세요."
DTYPE = "bfloat16" if torch.cuda.get_device_capability()[0] >= 8 else "float16"
print(torch.cuda.get_device_name(0), "→", DTYPE)

In [ ]:
# 5) 0.6B
!python -m scripts.plan_eval.run_quality --model Qwen/Qwen3-0.6B --dtype {DTYPE} --out out/plan_eval

In [ ]:
# 6) 1.7B
!python -m scripts.plan_eval.run_quality --model Qwen/Qwen3-1.7B --dtype {DTYPE} --out out/plan_eval

In [ ]:
# 7) 비교표
!python -m scripts.plan_eval.compare out/plan_eval/Qwen__Qwen3-0.6B__{DTYPE}.summary.json out/plan_eval/Qwen__Qwen3-1.7B__{DTYPE}.summary.json --out out/plan_eval/compare.md > /dev/null
from IPython.display import Markdown, display
display(Markdown(open('out/plan_eval/compare.md', encoding='utf-8').read()))

## 사람이 봐야 하는 것

표의 숫자만으로는 **요약이 사실과 맞는지**를 알 수 없다. `*.review.csv`를 열어 `요약_사실일치` 칸에 O/X를 적는다
(원문에 없는 내용이 들어갔거나 사실이 바뀌었으면 X). 두 모델 파일을 같이 보면 어느 쪽이 나은지 바로 보인다.

그 밖에 `*.records.jsonl`의 `raw_output`에서 모델이 실제로 낸 글을 볼 수 있다.

In [ ]:
# 8) 결과 내려받기 (zip 하나)
!cd /content/repo && zip -q -r plan_eval_results.zip out/plan_eval
from google.colab import files
files.download('/content/repo/plan_eval_results.zip')